In [9]:
import torch 
import torch.nn as nn
import pandas as pd 
import numpy as  np 


In [23]:
insurance_data = pd.read_csv('../supervised_learning__/insurance.csv')
from sklearn.preprocessing import OneHotEncoder, StandardScaler
X_encoding = insurance_data[insurance_data.select_dtypes(include='number').columns]

torch.manual_seed(42)
input = torch.tensor(np.array(X_encoding))

neuron = nn.Linear(in_features=4 ,out_features=1)

output = neuron(input)

RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float

In [28]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

insurance_data = pd.read_csv('../supervised_learning__/insurance.csv')

X_encoding = insurance_data.select_dtypes(include='number')

torch.manual_seed(42)

input = torch.tensor(X_encoding.to_numpy(), dtype=torch.float32)

neuron = nn.Linear(
    in_features=4,
    out_features=1
)

output = neuron(input)

print(input.shape)
print(output.shape)
print(output[:5])

print(neuron.weight)
print(neuron.bias)

torch.Size([1338, 4])
torch.Size([1338, 1])
tensor([[ 7774.0732],
        [  813.2247],
        [ 2067.6006],
        [10119.5195],
        [ 1800.1769]], grad_fn=<SliceBackward0>)
Parameter containing:
tensor([[ 0.3823,  0.4150, -0.1171,  0.4593]], requires_grad=True)
Parameter containing:
tensor([-0.1096], requires_grad=True)


In [30]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

insurance_data = pd.read_csv('../supervised_learning__/insurance.csv')

X = insurance_data.select_dtypes(include='number').drop(columns=['charges'])
y = insurance_data['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.float32).reshape(-1, 1)

torch.manual_seed(42)

neuron = nn.Linear(
    in_features=X_train.shape[1],
    out_features=1
)

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(neuron.parameters(), lr=0.01)

for epoch in range(1000):

    neuron.train()

    y_pred = neuron(X_train)

    loss = loss_fn(y_pred, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

neuron.eval()

with torch.no_grad():
    y_pred = neuron(X_test)

y_pred = y_pred.numpy().flatten()
y_test_np = y_test.numpy().flatten()

mae = mean_absolute_error(y_test_np, y_pred)
mse = mean_squared_error(y_test_np, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_np, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 9181.30078125
MSE : 131201328.0
RMSE: 11454.31482018894
R²  : 0.15489596128463745


In [64]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

insurance_data = pd.read_csv('../supervised_learning__/insurance.csv')

X = insurance_data.drop(columns=['charges'])
y = insurance_data['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

numeric_features = ['age', 'bmi', 'children']
categorical_features = ['sex', 'smoker', 'region']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(
    y_train.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

y_test = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(X_train.shape[1], 80),
    
    nn.ReLU(),
    # nn.Linear(64, 32),
    # nn.ReLU(),
    # nn.Linear(32, 16),
    # nn.ReLU(),
    # nn.Linear(16, 1)
    # nn.Linear(256,128),
    # nn.ReLU(),
    # nn.Linear(128, 64),
    # nn.ReLU(),
    nn.Linear(80,32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 3000

for epoch in range(epochs):

    model.train()

    y_pred = model(X_train)

    loss = loss_fn(y_pred, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 500 == 0:
        print(
            f"Epoch: {epoch + 1}, "
            f"Loss: {loss.item():.4f}"
        )

model.eval()

with torch.no_grad():
    y_pred = model(X_test)

y_pred = y_pred.numpy().flatten()
y_test_np = y_test.numpy().flatten()

mae = mean_absolute_error(y_test_np, y_pred)
mse = mean_squared_error(y_test_np, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_np, y_pred)

print("\nEvaluation")
print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R² :", r2)

Epoch: 500, Loss: 106828376.0000
Epoch: 1000, Loss: 43794120.0000
Epoch: 1500, Loss: 31810894.0000
Epoch: 2000, Loss: 26580594.0000
Epoch: 2500, Loss: 21696848.0000
Epoch: 3000, Loss: 20225220.0000

Evaluation
MAE : 2513.495361328125
MSE : 18447628.0
RMSE: 4295.070197330889
R² : 0.8811737298965454
